In [6]:
import pandas as pd
df=pd.read_csv("/content/IMDB Dataset.csv", encoding='latin1', on_bad_lines='skip', engine='python')

print(df.head())
print(df['sentiment'].value_counts())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
sentiment
positive    22577
negative    22518
Name: count, dtype: int64


In [7]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
45090,Let me start by saying how much I love the TV ...,negative
45091,Richard Attenborough who already given us magn...,positive
45092,A refreshing interview with the legendary Ital...,positive
45093,"While it's not ""perfect"", it's close. Love Bar...",positive


In [8]:
df['sentiment']=df['sentiment'].map({
    'positive':1,
    'negative':0
})

In [9]:
import re

negation_words =[
    "not good","not bad","not great","don't like","didn't like","never liker","wasn't good",
    "isn't good","no good"
]
def clean_text(text):
  text=text.lower()
  text=re.sub(r"[^a-aA-Z\s']"," ",text)

  for phrase in negation_words:
    text=text.replace(phrase,phrase.replace(" ","_"))
  return text

In [10]:
df['review']=df['review'].apply(clean_text)

In [11]:
from sklearn.model_selection import train_test_split

x_train,x_test,y_train,y_test =train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.2,
    random_state=42
)

In [12]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

vocab_size =20000
max_len=250

tokenizer =Tokenizer(num_words=vocab_size,oov_token="<OOV>")
tokenizer.fit_on_texts(x_train)

x_train_seq =tokenizer.texts_to_sequences(x_train)
x_test_seq =tokenizer.texts_to_sequences(x_test)

x_train_pad =pad_sequences(x_train_seq,maxlen=max_len,padding='post')
x_test_pad =pad_sequences(x_test_seq,maxlen=max_len,padding='post')

In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM, Dense, Dropout

model=Sequential([
    Embedding(vocab_size,128,input_length=max_len),

    LSTM(128,dropout=0.3,recurrent_dropout=0.3),

    Dense(64,activation='relu'),
    Dropout(0.3),

    Dense(1,activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
history=model.fit(
    x_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(x_test_pad, y_test)
)

Epoch 1/5
564/564 ━━━━━━━━━━━━━━━━━━━━ 436s 763ms/step - accuracy: 0.5070 - loss: 0.6932 - val_accuracy: 0.5043 - val_loss: 0.6933
Epoch 2/5
564/564 ━━━━━━━━━━━━━━━━━━━━ 434s 770ms/step - accuracy: 0.5017 - loss: 0.6933 - val_accuracy: 0.4991 - val_loss: 0.6930
Epoch 3/5
564/564 ━━━━━━━━━━━━━━━━━━━━ 434s 770ms/step - accuracy: 0.4987 - loss: 0.6931 - val_accuracy: 0.5042 - val_loss: 0.6930
Epoch 4/5
564/564 ━━━━━━━━━━━━━━━━━━━━ 448s 780ms/step - accuracy: 0.5047 - loss: 0.6928 - val_accuracy: 0.5044 - val_loss: 0.6929
Epoch 5/5
564/564 ━━━━━━━━━━━━━━━━━━━━ 433s 767ms/step - accuracy: 0.5018 - loss: 0.6929 - val_accuracy: 0.5046 - val_loss: 0.6931


In [17]:
loss,acc =model.evaluate(x_test_pad,y_test)
print("Test Accuracy:",acc)

282/282 ━━━━━━━━━━━━━━━━━━━━ 26s 92ms/step - accuracy: 0.5046 - loss: 0.6931
Test Accuracy: 0.5046014189720154


In [18]:
def predict_sentiment(review):
  review =clean_text(review)
  seq=tokenizer.texts_to_sequences([review])
  padded =pad_sequences(seq,maxlen=max_len,padding='post')
  prediction=model.predict(padded)[0][0]
  print("\nReview:",review)
  print("Score:",prediction)
  if prediction>=0.5:
    print("Sentiment: Positive")
  else:
    print("Sentiment: Negative")

In [19]:
predict_sentiment(
    "The movie was not good"
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 574ms/step

Review:            a          
Score: 0.49445036
Sentiment: Negative
